In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch import nn
import torch
from IPython.display import Markdown
import os

In [2]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## Category or class names

In [3]:
# Map labels to integers
categories=['負面','正面']

In [4]:

label_to_id = { cate : i for i, cate in enumerate(categories)}

In [5]:
label_to_id

{'負面': 0, '正面': 1}

In [6]:
id_to_label = { i : cate for i, cate in enumerate(categories)}

In [7]:
id_to_label

{0: '負面', 1: '正面'}

# import our custom QwenForClassifier

In [8]:
from custom_qwen_model import QwenForClassifier

# Load

In [9]:
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [11]:
# 載入基礎模型
full_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:

hidden_size = full_model.config.hidden_size

# 建立分類模型
model = QwenForClassifier(full_model.model, hidden_size, num_labels=2)


#
model_path = "trained_classifier_v4-5epochs-acc0.93"
# model_path = "checkpoints_v3\checkpoint-4145"
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

model.load_model(model_path, device=device)


# 載入分類器權重
# classifier_path = "trained_classifier_v4"
# classifier_weights_path = os.path.join(classifier_path, "classifier_weights.pt")
# if os.path.isfile(classifier_weights_path):
#     classifier_weights = torch.load(classifier_weights_path, map_location=device, weights_only=True)
#     model.classifier.load_state_dict(classifier_weights)
# else:
#     print(f"Warning: 在 {classifier_weights_path} 找不到分類器權重")
    
# 移動到指定設備
model = model.to(device)

已載入分類器權重: trained_classifier_v4-5epochs-acc0.93\classifier_weights.pt


In [13]:
full_model


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [14]:
model

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMS

In [15]:
model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
)

# 分類模型怎麼用?

In [16]:
import numpy as np
from transformers import AutoTokenizer

# Function to make predictions
def predict_sentiment(text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    logits = outputs["logits"]
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_label[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_label[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [17]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


{'text': '今天天氣真好，我很開心',
 'sentiment': '正面',
 'confidence': 1.0,
 'probabilities': {'負面': 0.0, '正面': 1.0}}

In [18]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 1.0,
 'probabilities': {'負面': 1.0, '正面': 0.0}}

In [19]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_sentiment(text, model, tokenizer, device)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'sentiment': '正面',
 'confidence': 1.0,
 'probabilities': {'負面': 0.0, '正面': 1.0}}

In [20]:
text = "我沒有很討厭這部電影"
predict_sentiment(text, model, tokenizer, device)

{'text': '我沒有很討厭這部電影',
 'sentiment': '負面',
 'confidence': 0.99,
 'probabilities': {'負面': 0.99, '正面': 0.01}}

In [21]:
text = "我沒有很討厭這部電影，但也不會推薦給朋友"
predict_sentiment(text, model, tokenizer, device)

{'text': '我沒有很討厭這部電影，但也不會推薦給朋友',
 'sentiment': '負面',
 'confidence': 0.98,
 'probabilities': {'負面': 0.98, '正面': 0.02}}

# 原始語言模型怎麼用?

In [22]:

#   generated_ids = model.generate(**model_inputs, 
#                                  max_new_tokens=512, 
#                                  do_sample=True, 
#                                  pad_token_id=tokenizer.eos_token_id)

def generate_text(input_prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [23]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 保持良好的生活习惯，包括规律的作息时间、均衡饮食和适量运动。
2. 培养积极的心态，避免过度的压力和焦虑，通过冥想、呼吸练习等方式缓解压力。
3. 定期进行健康检查，及时发现并解决身体上的问题，同时关注心理健康，建立良好的社交网络。

In [24]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: total: 1min 12s
Wall time: 18.8 s


1. 調整能源使用，選擇更節能的能源設備和燃料；
2. 增加公共交通工具，減少私家車的使用；
3. 遵守環保法規，不進行非法排污行為；
4. 應用智能交通系統，減少車輛排放；
5. 對於工業排放，采用低排放、高效能的技術；
6. 市場監管部門應對污染企業進行監督和處罰；
7. 降低消費習慣，減少使用一次性塑料制品。

In [25]:
%%time
text="亞洲最高的山?"
result = generate_text(text)
Markdown(result)

CPU times: total: 20.5 s
Wall time: 5.74 s


珠穆朗玛峰是亚洲最高的山峰，海拔8,848米（29,029英尺）。

In [26]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [27]:
stop here!

SyntaxError: invalid syntax (2745754519.py, line 1)

# views.py

In [ ]:
from django.shortcuts import render
from django.views.decorators.csrf import csrf_exempt
from django.http import JsonResponse
import json  
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import markdown

from .custom_qwen_model import QwenForClassifier

# If we don't use GPU, we can set the environment variable to disable it
# os.environ['CUDA_VISIBLE_DEVICES'] = '-1'


# Loading app large language model, news classifier and sentiment classifier

# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


# 根據設備選擇適當的資料類型
dtype = torch.float16 if device.type == 'cuda' else torch.float32
print(f"使用資料類型: {dtype}")

# Map labels to integers
sentiment_categories=['負面','正面']
sentimentlabel_to_id = { cate : i for i, cate in enumerate(sentiment_categories)}
id_to_sentimentlabel = { i : cate for i, cate in enumerate(sentiment_categories)}

# Convert news category name ('政治','科技','運動',...) into number (0,1,2,...)
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']
newslabel_to_id = { cate : i for i, cate in enumerate(news_categories)}
id_to_newslabel = { i : cate for i, cate in enumerate(news_categories)}

# Tokenizer initialization
model_id = "Qwen/Qwen2.5-0.5B-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Model initialization
full_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
#full_model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=dtype).to(device)
hidden_size = full_model.config.hidden_size

# Sentiment classifier model initialization
model_sentiment_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels=len(sentiment_categories))
model_path_sentiment = "app_llm_classifier/trained_models/trained_sentiment_classifier_v4-5epochs-acc0.93"
model_sentiment_classifier.load_model(model_path_sentiment, device=device)
model_sentiment_classifier = model_sentiment_classifier.to(device) # 移動到設備
#model_sentiment_classifier = model_sentiment_classifier.to(device, dtype=dtype) # 移動到設備

# News classifier model initialization
model_news_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels=len(news_categories))
model_path_news = "app_llm_classifier/trained_models/trained_news_classifier_v3-6epochs-acc0.90"
model_news_classifier.load_model(model_path_news, device=device)
#model_news_classifier = model_news_classifier.to(device, dtype=dtype)
model_news_classifier = model_news_classifier.to(device)

# Function to make sentiment predictions
def predict_sentiment(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_sentiment_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    logits = outputs["logits"]  
    probabilities = F.softmax(logits, dim=-1)
    
    # Get the predicted class and label
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    predicted_label = id_to_sentimentlabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence, 2),
        "probabilities": {
            id_to_sentimentlabel[i]: round(prob.item(), 2) for i, prob in enumerate(probabilities[0])
        }
    }

# Function to make news category predictions
def predict_news_category(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_news_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    logits = outputs["logits"]
    probabilities = F.softmax(logits, dim=-1)
    
    # Get the predicted class and label
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    predicted_label = id_to_newslabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence, 2),
        "probabilities": {
            id_to_newslabel[i]: round(prob.item(), 2) for i, prob in enumerate(probabilities[0])
        }
    }

# Generate text using the LLM
def generate_text(messages):
    """
    Generate response text using the model.
    """
    text_chat_template = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    print("text_chat_templte:", text_chat_template)
    
    model_inputs = tokenizer([text_chat_template], return_tensors="pt").to(device)
    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response

# Function to handle sentiment prediction
def home_sentiment(request):
    return render(request, "app_llm_classifier/home-sentiment.html")

@csrf_exempt
def api_get_sentiment(request):
    input_text = request.POST.get('input_text')
    print(input_text)
    print(request.content_type)
    print(request.body)

    sentiment_prob = predict_sentiment(input_text)
    return JsonResponse(sentiment_prob)

# Function to handle news category prediction
def home_news_category(request):
    return render(request, "app_llm_classifier/home-news-category.html")

@csrf_exempt
def api_get_news_category(request):
    input_text = request.POST.get('input_text')
    response = predict_news_category(input_text)
    return JsonResponse(response)

# Function to handle text generation using LLM
def home_chatbot(request):
    return render(request, "app_llm_classifier/home-text-generation.html")

@csrf_exempt
def api_get_llm_response(request):
    input_text = request.POST.get('input_text')
    conversation_history_json = request.POST.get('conversation_history')
    
    print("input_text", input_text)
    print("conversation_history_json", conversation_history_json)
    
    # Process conversation history if available
    conversation_history = []
    if conversation_history_json:
        try:
            conversation_history = json.loads(conversation_history_json)
        except json.JSONDecodeError:
            print("Error parsing conversation history JSON")
            conversation_history = []
    
    # Prepare messages for the model
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]
    
    # Add conversation history if provided
    if conversation_history:
        messages.extend(conversation_history)
    
    # Always add the current user message
    messages.append({"role": "user", "content": input_text})
    
    print("messages:", messages)
    
    # Generate response with the prepared text
    response = generate_text(messages)
    print("response:", response)
    
    # Convert the string response into a dictionary format
    response_dict = {
        "response": response,
        "input": input_text
    }

    return JsonResponse(response_dict)


def model_introduction(request):
    # Read the markdown file
    markdown_file_path = 'app_llm_classifier/markdown-files/model-introduction.md'
    # Read the markdown file
    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        markdown_content = f.read()
    
    # Convert markdown to HTML
    html_content = markdown.markdown(markdown_content, extensions=['fenced_code', 'codehilite'])
    
    # Pass the HTML content to the template
    context = {
        'html_content': html_content
    }
    
    return render(request, "app_llm_classifier/model-introduction.html", context)

print("Loading app large language model, news classifier and sentiment classifier OK.")
